# Chunking Strategies for Code Files
*Part of the Code Review Agent capstone — this notebook determines the chunking strategy for ingesting Python best-practice docs into ChromaDB.*

## Problem Statement
Compare fixed-size, recursive, and AST-aware chunking on a Python source file to determine the best strategy for the Code Review Agent's ingestion pipeline.

## My Approach
I expected AST-aware chunking to win from the start. 
* Fixed-size splitting is essentially blind — it has no idea what Python even is, it just counts characters. 
* Recursive is smarter about whitespace boundaries but still can't tell a function signature from a comment. 
* AST actually *parses* the code into a syntax tree, so every chunk it produces is a valid, complete semantic unit — a function or a method. For a Code Review Agent that asks "is this function handling exceptions correctly?", you need the full function in one chunk, not half of it.
 
## What I Learned
* The main thing: chunk boundaries are a retrieval quality problem, not just a storage problem. 
* A chunk that cuts mid-function doesn't just waste space — it actively misleads the LLM, because it retrieves incomplete context that *looks* complete.
* The mid-cut detection check (`"def " in chunk but doesn't start with def`) made this concrete — both fixed-size and recursive produced at least one cut like this on a file that's under 800 characters. On real codebases, it would be worse.
* Also learned that `ast.walk()` vs `ast.iter_child_nodes()` is a meaningful choice — `walk()` visits the entire subtree recursively, which caused double-counting of methods that already appeared inside the class node.
 
## Where I Got Stuck
* The AST chunker produced 6 chunks when I expected 4 or 5. Printing them revealed the issue: `ast.walk()` returns both `DataPipeline` (the full class) *and* `__init__`, `run`, `report` as separate nodes — so the methods appeared twice. Switching to `iter_child_nodes()` on the tree for top-level nodes, then `iter_child_nodes()` on each `ClassDef` for methods, fixed the duplication cleanly.
 
## What I'd Do Differently
* Add a visual "chunk boundary" print — something that shows each chunk with its index and first/last line, not just aggregate stats. 
* The table tells you *that* a mid-cut happened, but not *where*. In a debugging session that would've saved time. Also should have added the decision markdown cell — which strategy I'm carrying forward into the Code Review Agent and why.

## My Solution (Naive)
*First attempt — written before reviewing feedback*

In [3]:
import ast
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [4]:
SAMPLE_CODE = '''
import os
import json

def load_config(path: str) -> dict:
    """Load JSON config from disk."""
    with open(path, "r") as f:
        return json.load(f)

def validate_schema(data: dict, required_keys: list) -> bool:
    """Check that all required keys exist in data dict."""
    for key in required_keys:
        if key not in data:
            print(f"Missing key: {key}")
            return False
    return True

class DataPipeline:
    def __init__(self, config_path: str):
        self.config = load_config(config_path)
        self.errors = []

    def run(self, records: list) -> list:
        """Process records, skip invalid ones."""
        results = []
        for record in records:
            if validate_schema(record, ["id", "value"]):
                results.append(record)
            else:
                self.errors.append(record)
        return results

    def report(self) -> None:
        print(f"Pipeline complete. Errors: {len(self.errors)}")
'''

In [37]:
# --- Strategy 1: Fixed Size Splitting
# Uses a strict character count with overlap to prevent loss at boundaries 
fixed_splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50)
fixed_chunks = fixed_splitter.split_text(SAMPLE_CODE)

for i, chunk in enumerate(fixed_chunks):
    print(f"{'='*10} Chunk {i+1} {'='*20}")
    print(chunk)

Created a chunk of size 323, which is longer than the specified 300


========== Chunk 1 ====================
import os
import json

def load_config(path: str) -> dict:
    """Load JSON config from disk."""
    with open(path, "r") as f:
        return json.load(f)
========== Chunk 2 ====================
def validate_schema(data: dict, required_keys: list) -> bool:
    """Check that all required keys exist in data dict."""
    for key in required_keys:
        if key not in data:
            print(f"Missing key: {key}")
            return False
    return True
========== Chunk 3 ====================
class DataPipeline:
    def __init__(self, config_path: str):
        self.config = load_config(config_path)
        self.errors = []
========== Chunk 4 ====================
def run(self, records: list) -> list:
        """Process records, skip invalid ones."""
        results = []
        for record in records:
            if validate_schema(record, ["id", "value"]):
                results.append(record)
            else:
                self.errors.append(

In [40]:
# --- Strategy 2: Recursive Splitting 
# Better for prose/code as it respects logical separators (\n\n, \n) 
recursive_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
recursive_chunks = recursive_splitter.split_text(SAMPLE_CODE)

for i, chunk in enumerate(recursive_chunks):
    print(f"{'='*10} Chunk {i+1} {'='*20}")
    print(chunk)

========== Chunk 1 ====================
import os
import json

def load_config(path: str) -> dict:
    """Load JSON config from disk."""
    with open(path, "r") as f:
        return json.load(f)
========== Chunk 2 ====================
def validate_schema(data: dict, required_keys: list) -> bool:
    """Check that all required keys exist in data dict."""
    for key in required_keys:
        if key not in data:
            print(f"Missing key: {key}")
            return False
    return True
========== Chunk 3 ====================
class DataPipeline:
    def __init__(self, config_path: str):
        self.config = load_config(config_path)
        self.errors = []
========== Chunk 4 ====================
def run(self, records: list) -> list:
        """Process records, skip invalid ones."""
        results = []
        for record in records:
            if validate_schema(record, ["id", "value"]):
                results.append(record)
            else:
========== Chunk 5 ================

In [39]:
# --- Strategy 3: AST-Aware Splitting --- 
# Walks the syntax tree to extract complete functions and classes 
tree = ast.parse(SAMPLE_CODE)
ast_chunks = []

for node in ast.walk(tree):
    # print(node) 
    # Extract only function and class definitions
    if isinstance(node, (ast.FunctionDef, ast.ClassDef)):
        # Must pass original raw_code as the first argument 
        segment = ast.get_source_segment(SAMPLE_CODE, node)
        if segment:
            #print(segment)
            ast_chunks.append(segment)


for i, chunk in enumerate(ast_chunks):
    print(f"{'='*10} Chunk {i+1} {'='*20}")
    print(chunk)

========== Chunk 1 ====================
def load_config(path: str) -> dict:
    """Load JSON config from disk."""
    with open(path, "r") as f:
        return json.load(f)
========== Chunk 2 ====================
def validate_schema(data: dict, required_keys: list) -> bool:
    """Check that all required keys exist in data dict."""
    for key in required_keys:
        if key not in data:
            print(f"Missing key: {key}")
            return False
    return True
========== Chunk 3 ====================
class DataPipeline:
    def __init__(self, config_path: str):
        self.config = load_config(config_path)
        self.errors = []

    def run(self, records: list) -> list:
        """Process records, skip invalid ones."""
        results = []
        for record in records:
            if validate_schema(record, ["id", "value"]):
                results.append(record)
            else:
                self.errors.append(record)
        return results

    def report(self) -> No

In [31]:
# --- Evaluation & Validation ---
def evaluate_strategy(name, chunks):
    lengths = [len(c) for c in chunks]
    # Check for "mid-function cuts": 'def' appears but not at the start 
    mid_cuts = sum(1 for c in chunks if "def " in c and not c.strip().startswith(("def ", "class ")))
    
    print(f"{name:<15}| {len(chunks):<6} | {round(sum(lengths)/len(lengths),1):<7} | {min(lengths):<7} | {max(lengths):<7} | {mid_cuts:<13}")


In [32]:
print(f"{'Strategy':<15}| Chunks | Avg Len | Min Len | Max Len | Mid-def cuts?")
print(f"{'-'*15}+{'-'*8}+{'-'*9}+{'-'*9}+{'-'*9}+{'-'*15}")
evaluate_strategy("Fixed Size", fixed_chunks)
evaluate_strategy("Recursive", recursive_chunks)
evaluate_strategy("AST-Aware", ast_chunks)


Strategy       | Chunks | Avg Len | Min Len | Max Len | Mid-def cuts?
---------------+--------+---------+---------+---------+---------------
Fixed Size     | 5      | 191.2   | 89      | 319     | 1            
Recursive      | 6      | 160.2   | 71      | 260     | 1            
AST-Aware      | 6      | 243.7   | 89      | 553     | 0            


## Refactored Solution

In [ ]:
# --- Strategy 3: AST-Aware Splitting ---
# Extracts top-level functions + individual class methods as separate chunks
# Avoids double-counting by using iter_child_nodes instead of ast.walk

tree = ast.parse(SAMPLE_CODE)
refactored_ast_chunks = []

for node in ast.iter_child_nodes(tree):
    # print(node)
    if isinstance(node, ast.FunctionDef):
        segment = ast.get_source_segment(SAMPLE_CODE, node)
        if segment:
            refactored_ast_chunks.append(segment)
    elif isinstance(node, ast.ClassDef):
        # Each method is its own chunk — class body is not stored as one blob
        for child in ast.iter_child_nodes(node):
            # print(" -> ",child)
            if isinstance(child, ast.FunctionDef):
                segment = ast.get_source_segment(SAMPLE_CODE, child)
                if segment:
                    refactored_ast_chunks.append(segment)

 ->  <ast.FunctionDef object at 0x000001AE03A22210>
 ->  <ast.FunctionDef object at 0x000001AE03A37E90>
 ->  <ast.FunctionDef object at 0x000001AE03B17650>


In [46]:
for i, chunk in enumerate(refactored_ast_chunks):
    print(f"{'='*10} Chunk {i+1} {'='*20}")
    print(chunk)

========== Chunk 1 ====================
def load_config(path: str) -> dict:
    """Load JSON config from disk."""
    with open(path, "r") as f:
        return json.load(f)
========== Chunk 2 ====================
def validate_schema(data: dict, required_keys: list) -> bool:
    """Check that all required keys exist in data dict."""
    for key in required_keys:
        if key not in data:
            print(f"Missing key: {key}")
            return False
    return True
========== Chunk 3 ====================
def __init__(self, config_path: str):
        self.config = load_config(config_path)
        self.errors = []
========== Chunk 4 ====================
def run(self, records: list) -> list:
        """Process records, skip invalid ones."""
        results = []
        for record in records:
            if validate_schema(record, ["id", "value"]):
                results.append(record)
            else:
                self.errors.append(record)
        return results
========== Chunk

In [47]:
print(f"{'Strategy':<15}| Chunks | Avg Len | Min Len | Max Len | Mid-def cuts?")
print(f"{'-'*15}+{'-'*8}+{'-'*9}+{'-'*9}+{'-'*9}+{'-'*15}")
evaluate_strategy("Fixed Size", fixed_chunks)
evaluate_strategy("Recursive", recursive_chunks)
evaluate_strategy("AST-Aware", refactored_ast_chunks)


Strategy       | Chunks | Avg Len | Min Len | Max Len | Mid-def cuts?
---------------+--------+---------+---------+---------+---------------
Fixed Size     | 5      | 191.2   | 89      | 319     | 1            
Recursive      | 6      | 160.2   | 71      | 260     | 1            
AST-Aware      | 5      | 181.8   | 89      | 319     | 0            
